# Qwen3 4B ASR Embedding Server — Kaggle T4×2

**Output notebook:** `qwen3_4b_server_v1.ipynb`

This notebook is designed for the `zintomvn/Multimodal-Retrieval` retrieval stack at Git commit
`f1ebe488da60a950711ea62401b180a144fc4d26`.

## Important model choice

For **ASR semantic retrieval**, this notebook serves **`Qwen/Qwen3-Embedding-4B`**, not
`Qwen/Qwen3-VL-4B-Instruct`.

- `Qwen3-VL-4B-Instruct` is an image/text-to-text VLM intended for generation and multimodal reasoning.
- `Qwen3-Embedding-4B` is the Qwen 4B text embedding model intended for retrieval.
- Official embedding dimension: **2560**.
- This matches the existing ASR embedding notebook pattern and the repository's OpenAI-compatible
  `/v1/models` + `/v1/embeddings` contract.

References:
- https://huggingface.co/Qwen/Qwen3-Embedding-4B
- https://huggingface.co/Qwen/Qwen3-VL-4B-Instruct
- Repository: https://github.com/zintomvn/Multimodal-Retrieval

## Repository integration points used

The current repository already:
1. builds ASR/text records,
2. stores text vectors in Milvus,
3. has an OpenAI-compatible embedding endpoint contract,
4. searches embeddings by calling `POST <base_url>/embeddings`.

The new server keeps that API shape and returns L2-normalized float vectors.


## Architecture

```text
Web / FastAPI backend
        |
        | POST /v1/embeddings
        v
+-----------------------------+
| FastAPI coordinator         |  uvicorn workers = 1
| - auth / validation         |
| - bounded request queue     |
| - dynamic micro-batching    |
| - result dispatcher         |
+-------------+---------------+
              |
       shared MP queue
       /              \
      v                v
+-----------+      +-----------+
| GPU proc0 |      | GPU proc1 |
| cuda:0 T4 |      | cuda:1 T4 |
| Qwen3 4B  |      | Qwen3 4B  |
+-----------+      +-----------+
```

Why not `uvicorn --workers 8`?

Each Uvicorn process would independently load a 4B model and multiply VRAM usage. Instead, HTTP concurrency
is handled asynchronously in one coordinator while exactly one inference process is pinned to each GPU.

The server performs **dynamic batching across separate incoming requests**. Query requests and document
requests are batched separately because Qwen3 retrieval queries should use the query instruction/prompt,
while indexed ASR documents should not.


In [ ]:
# 1) Install runtime dependencies.
# The cell first looks for offline wheels under /kaggle/input. If not found, it uses normal pip.
import os, sys, subprocess
from pathlib import Path

REQS = [
    "transformers>=4.51,<5",
    "sentence-transformers>=3.4,<6",
    "accelerate>=1,<2",
    "fastapi>=0.115,<1",
    "uvicorn[standard]>=0.30,<1",
    "orjson>=3.10,<4",
    "httpx>=0.27,<1",
    "pyngrok>=7,<8",
    "pymilvus>=2.5,<3",
]

wheel_dirs = sorted({
    str(p.parent)
    for p in Path("/kaggle/input").rglob("*.whl")
    if any(k in p.name.lower().replace("_", "-") for k in (
        "sentence-transformers", "transformers-", "accelerate-", "fastapi-", "uvicorn-", "orjson-", "httpx-"
    ))
}) if Path("/kaggle/input").exists() else []

cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"]
if wheel_dirs:
    print("Offline wheel directories detected:", wheel_dirs[:10])
    cmd += ["--no-index"]
    for d in wheel_dirs:
        cmd += ["--find-links", d]
cmd += REQS

print("Running pip install...")
subprocess.run(cmd, check=True)
print("Dependencies ready.")


In [ ]:
# 2) Hardware and environment check.
import json
import os
import platform
import subprocess
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("CPU count:", os.cpu_count())

if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU accelerator in Kaggle.")
if torch.cuda.device_count() < 2:
    raise RuntimeError(
        f"Expected Kaggle T4x2 but only found {torch.cuda.device_count()} GPU(s). "
        "Select the dual-T4 accelerator before starting the server."
    )

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print({
        "gpu": i,
        "name": p.name,
        "total_memory_GiB": round(p.total_memory / 1024**3, 2),
        "compute_capability": f"{p.major}.{p.minor}",
    })

try:
    print(subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free,utilization.gpu",
         "--format=csv,noheader,nounits"],
        text=True
    ))
except Exception as exc:
    print("nvidia-smi query skipped:", exc)


In [ ]:
# 3) Resolve Qwen3-Embedding-4B model and server settings.
import os
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-Embedding-4B"
EXPECTED_DIM = 2560
PORT = 8001

# Exact local path used by the ASR embedding notebook if this Kaggle Model is attached.
preferred_paths = [
    Path("/kaggle/input/models/annguyentranthien21/qwen3-embedding-4b/transformers/default/1"),
]

def looks_like_qwen3_embedding_4b(path: Path) -> bool:
    label = path.as_posix().lower().replace("_", "-")
    if not path.is_dir():
        return False
    if not (path / "config.json").exists():
        return False
    return "qwen3" in label and "embedding" in label and "4b" in label

def resolve_model_source() -> str:
    explicit = os.getenv("QWEN_EMBED_MODEL_SOURCE", "").strip()
    if explicit:
        return explicit

    for p in preferred_paths:
        if looks_like_qwen3_embedding_4b(p):
            return str(p)

    if Path("/kaggle/input").exists():
        candidates = []
        for cfg in Path("/kaggle/input").rglob("config.json"):
            p = cfg.parent
            if looks_like_qwen3_embedding_4b(p):
                candidates.append(p)
        candidates = sorted(set(candidates), key=lambda p: (len(p.parts), str(p)))
        if candidates:
            print("Auto-detected local model candidates:", candidates[:10])
            return str(candidates[0])

    # Falls back to Hugging Face Hub. Kaggle Internet must be enabled for this path.
    return MODEL_ID

MODEL_SOURCE = resolve_model_source()
print("MODEL_ID:", MODEL_ID)
print("MODEL_SOURCE:", MODEL_SOURCE)
print("EXPECTED_DIM:", EXPECTED_DIM)

# Throughput defaults tuned conservatively for 2xT4.
os.environ["QWEN_EMBED_MODEL_ID"] = MODEL_ID
os.environ["QWEN_EMBED_MODEL_SOURCE"] = MODEL_SOURCE
os.environ["QWEN_EMBED_DIM"] = str(EXPECTED_DIM)
os.environ["QWEN_EMBED_MAX_SEQ_LEN"] = os.getenv("QWEN_EMBED_MAX_SEQ_LEN", "2048")
os.environ["QWEN_GPU_BATCH_TEXTS"] = os.getenv("QWEN_GPU_BATCH_TEXTS", "8")
os.environ["QWEN_BATCH_WAIT_MS"] = os.getenv("QWEN_BATCH_WAIT_MS", "8")
os.environ["QWEN_MAX_QUEUE_JOBS"] = os.getenv("QWEN_MAX_QUEUE_JOBS", "1024")
os.environ["QWEN_MAX_TEXTS_PER_REQUEST"] = os.getenv("QWEN_MAX_TEXTS_PER_REQUEST", "128")
os.environ["QWEN_REQUEST_TIMEOUT_S"] = os.getenv("QWEN_REQUEST_TIMEOUT_S", "60")
os.environ["EMBED_HOST"] = "0.0.0.0"
os.environ["EMBED_PORT"] = str(PORT)

# Public tunnel protection. Best practice: save EMBED_API_KEY as a Kaggle Secret.
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    if not os.getenv("EMBED_API_KEY"):
        try:
            os.environ["EMBED_API_KEY"] = secrets.get_secret("EMBED_API_KEY")
        except Exception:
            pass
except Exception:
    pass

print({
    "max_seq_length": os.environ["QWEN_EMBED_MAX_SEQ_LEN"],
    "gpu_batch_texts": os.environ["QWEN_GPU_BATCH_TEXTS"],
    "batch_wait_ms": os.environ["QWEN_BATCH_WAIT_MS"],
    "queue_jobs": os.environ["QWEN_MAX_QUEUE_JOBS"],
    "api_key_enabled": bool(os.getenv("EMBED_API_KEY", "").strip()),
})


In [ ]:
# 4) Materialize the dual-GPU FastAPI server as a normal .py module.
# Using a real Python module makes multiprocessing "spawn" reliable in Jupyter/Kaggle.
from pathlib import Path

SERVER_PATH = Path("/kaggle/working/qwen3_4b_embedding_server.py")
SERVER_CODE = '\nfrom __future__ import annotations\n\nimport asyncio\nimport concurrent.futures\nimport gc\nimport multiprocessing as mp\nimport os\nimport queue\nimport threading\nimport time\nimport uuid\nfrom pathlib import Path\nfrom typing import Any, Literal\n\nimport numpy as np\nimport torch\nimport uvicorn\nfrom fastapi import Depends, FastAPI, Header, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import ORJSONResponse\nfrom pydantic import BaseModel\nfrom sentence_transformers import SentenceTransformer\n\nMODEL_ID = os.getenv("QWEN_EMBED_MODEL_ID", "Qwen/Qwen3-Embedding-4B")\nMODEL_SOURCE = os.getenv("QWEN_EMBED_MODEL_SOURCE", MODEL_ID)\nEMBEDDING_DIM = int(os.getenv("QWEN_EMBED_DIM", "2560"))\nMAX_SEQ_LENGTH = int(os.getenv("QWEN_EMBED_MAX_SEQ_LEN", "2048"))\nGPU_BATCH_TEXTS = int(os.getenv("QWEN_GPU_BATCH_TEXTS", "8"))\nDYNAMIC_BATCH_WAIT_MS = float(os.getenv("QWEN_BATCH_WAIT_MS", "8"))\nMAX_TEXTS_PER_REQUEST = int(os.getenv("QWEN_MAX_TEXTS_PER_REQUEST", "128"))\nMAX_TEXT_CHARS = int(os.getenv("QWEN_MAX_TEXT_CHARS", "24000"))\nMAX_QUEUE_JOBS = int(os.getenv("QWEN_MAX_QUEUE_JOBS", "1024"))\nREQUEST_TIMEOUT_S = float(os.getenv("QWEN_REQUEST_TIMEOUT_S", "60"))\nAPI_KEY = os.getenv("EMBED_API_KEY", "").strip()\nCPU_COUNT = os.cpu_count() or 2\n\n_GPU_COUNT = torch.cuda.device_count()\nGPU_IDS = list(range(_GPU_COUNT))\nif not GPU_IDS:\n    raise RuntimeError("No CUDA GPU detected. This server is intended for Kaggle T4x2.")\nif _GPU_COUNT < 2:\n    print(f"WARNING: only {_GPU_COUNT} GPU(s) detected; server will run with {_GPU_COUNT} GPU worker(s).")\n\nCPU_THREADS_PER_GPU = max(1, CPU_COUNT // max(1, len(GPU_IDS)))\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "true")\nos.environ.setdefault("RAYON_NUM_THREADS", str(CPU_THREADS_PER_GPU))\nos.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS_PER_GPU))\nos.environ.setdefault("MKL_NUM_THREADS", str(CPU_THREADS_PER_GPU))\n\nclass EmbeddingRequest(BaseModel):\n    model: str | None = None\n    input: str | list[str]\n    input_type: Literal["query", "document"] = "query"\n\ndef _as_texts(value: str | list[str]) -> list[str]:\n    texts = [value] if isinstance(value, str) else [str(x) for x in value]\n    texts = [t.strip() for t in texts]\n    if not texts or any(not t for t in texts):\n        raise HTTPException(status_code=400, detail="input must contain non-empty text")\n    if len(texts) > MAX_TEXTS_PER_REQUEST:\n        raise HTTPException(status_code=400, detail=f"too many texts: {len(texts)} > {MAX_TEXTS_PER_REQUEST}")\n    too_long = [i for i, t in enumerate(texts) if len(t) > MAX_TEXT_CHARS]\n    if too_long:\n        raise HTTPException(status_code=400, detail=f"text at index {too_long[0]} exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}")\n    return texts\n\ndef _load_model(gpu_id: int) -> SentenceTransformer:\n    torch.cuda.set_device(gpu_id)\n    torch.set_num_threads(CPU_THREADS_PER_GPU)\n    device = f"cuda:{gpu_id}"\n    kwargs: dict[str, Any] = {\n        "device": device,\n        "model_kwargs": {"torch_dtype": torch.float16},\n        "tokenizer_kwargs": {"padding_side": "left"},\n    }\n    if Path(MODEL_SOURCE).exists():\n        kwargs["local_files_only"] = True\n    model = SentenceTransformer(MODEL_SOURCE, **kwargs)\n    model.max_seq_length = MAX_SEQ_LENGTH\n    model.eval()\n\n    dim = int(model.get_sentence_embedding_dimension())\n    if dim != EMBEDDING_DIM:\n        raise RuntimeError(f"Embedding dim mismatch on GPU {gpu_id}: got {dim}, expected {EMBEDDING_DIM}")\n\n    # Warm both document and query paths.\n    model.encode(\n        ["kiểm tra hệ thống"],\n        batch_size=1,\n        normalize_embeddings=True,\n        convert_to_numpy=True,\n        show_progress_bar=False,\n    )\n    model.encode(\n        ["tìm đoạn video có người đang nói"],\n        prompt_name="query",\n        batch_size=1,\n        normalize_embeddings=True,\n        convert_to_numpy=True,\n        show_progress_bar=False,\n    )\n    torch.cuda.synchronize(gpu_id)\n    return model\n\ndef _encode_with_oom_fallback(model: SentenceTransformer, texts: list[str], input_type: str) -> np.ndarray:\n    kwargs: dict[str, Any] = {\n        "batch_size": min(GPU_BATCH_TEXTS, max(1, len(texts))),\n        "normalize_embeddings": True,\n        "convert_to_numpy": True,\n        "show_progress_bar": False,\n    }\n    if input_type == "query":\n        kwargs["prompt_name"] = "query"\n\n    try:\n        arr = model.encode(texts, **kwargs)\n        return np.asarray(arr, dtype=np.float32)\n    except torch.cuda.OutOfMemoryError:\n        if len(texts) <= 1:\n            raise\n        torch.cuda.empty_cache()\n        mid = len(texts) // 2\n        left = _encode_with_oom_fallback(model, texts[:mid], input_type)\n        right = _encode_with_oom_fallback(model, texts[mid:], input_type)\n        return np.concatenate([left, right], axis=0)\n\ndef gpu_worker_main(gpu_id: int, request_q: mp.Queue, result_q: mp.Queue) -> None:\n    worker_name = f"gpu-{gpu_id}"\n    try:\n        model = _load_model(gpu_id)\n        result_q.put({"kind": "worker_ready", "worker": worker_name, "gpu_id": gpu_id})\n    except Exception as exc:\n        result_q.put({"kind": "worker_failed", "worker": worker_name, "gpu_id": gpu_id, "error": repr(exc)})\n        return\n\n    carry = None\n    stop_after_batch = False\n    while True:\n        first = carry if carry is not None else request_q.get()\n        carry = None\n        if first is None:\n            break\n\n        jobs = [first]\n        total_texts = len(first["texts"])\n        deadline = time.perf_counter() + DYNAMIC_BATCH_WAIT_MS / 1000.0\n\n        while total_texts < GPU_BATCH_TEXTS:\n            remaining = deadline - time.perf_counter()\n            if remaining <= 0:\n                break\n            try:\n                nxt = request_q.get(timeout=remaining)\n            except queue.Empty:\n                break\n            if nxt is None:\n                stop_after_batch = True\n                break\n            if nxt["input_type"] != first["input_type"]:\n                carry = nxt\n                break\n            if total_texts + len(nxt["texts"]) > GPU_BATCH_TEXTS:\n                carry = nxt\n                break\n            jobs.append(nxt)\n            total_texts += len(nxt["texts"])\n\n        flat: list[str] = []\n        spans: list[tuple[str, int, int]] = []\n        cursor = 0\n        for job in jobs:\n            flat.extend(job["texts"])\n            spans.append((job["id"], cursor, cursor + len(job["texts"])))\n            cursor += len(job["texts"])\n\n        started = time.perf_counter()\n        try:\n            vectors = _encode_with_oom_fallback(model, flat, first["input_type"])\n            if vectors.shape != (len(flat), EMBEDDING_DIM):\n                raise RuntimeError(f"bad embedding shape {vectors.shape}")\n            elapsed_ms = (time.perf_counter() - started) * 1000.0\n            for req_id, lo, hi in spans:\n                result_q.put({\n                    "kind": "result",\n                    "id": req_id,\n                    "vectors": vectors[lo:hi].tolist(),\n                    "worker": worker_name,\n                    "batch_texts": len(flat),\n                    "gpu_ms": elapsed_ms,\n                })\n        except Exception as exc:\n            for req_id, _, _ in spans:\n                result_q.put({"kind": "error", "id": req_id, "worker": worker_name, "error": repr(exc)})\n            gc.collect()\n            torch.cuda.empty_cache()\n\n        if stop_after_batch:\n            break\n\nclass Coordinator:\n    def __init__(self) -> None:\n        self.ctx = mp.get_context("spawn")\n        self.request_q: mp.Queue = self.ctx.Queue(maxsize=MAX_QUEUE_JOBS)\n        self.result_q: mp.Queue = self.ctx.Queue(maxsize=MAX_QUEUE_JOBS * 2)\n        self.pending: dict[str, concurrent.futures.Future] = {}\n        self.lock = threading.Lock()\n        self.workers: list[mp.Process] = []\n        self.ready_gpus: set[int] = set()\n        self.failed_workers: dict[int, str] = {}\n        self.started_at = time.time()\n        self.total_requests = 0\n        self.total_texts = 0\n        self._dispatcher = threading.Thread(target=self._dispatch_results, daemon=True)\n\n    def start(self) -> None:\n        for gpu_id in GPU_IDS:\n            p = self.ctx.Process(\n                target=gpu_worker_main,\n                args=(gpu_id, self.request_q, self.result_q),\n                daemon=True,\n            )\n            p.start()\n            self.workers.append(p)\n        self._dispatcher.start()\n\n    def _dispatch_results(self) -> None:\n        while True:\n            msg = self.result_q.get()\n            kind = msg.get("kind")\n            if kind == "worker_ready":\n                self.ready_gpus.add(int(msg["gpu_id"]))\n                continue\n            if kind == "worker_failed":\n                self.failed_workers[int(msg["gpu_id"])] = str(msg["error"])\n                continue\n\n            req_id = msg.get("id")\n            if not req_id:\n                continue\n            with self.lock:\n                fut = self.pending.pop(req_id, None)\n            if fut is None or fut.done():\n                continue\n            if kind == "result":\n                fut.set_result(msg)\n            else:\n                fut.set_exception(RuntimeError(str(msg.get("error", "worker error"))))\n\n    async def submit(self, texts: list[str], input_type: str) -> dict[str, Any]:\n        req_id = uuid.uuid4().hex\n        fut: concurrent.futures.Future = concurrent.futures.Future()\n        with self.lock:\n            self.pending[req_id] = fut\n\n        job = {"id": req_id, "texts": texts, "input_type": input_type}\n        try:\n            self.request_q.put_nowait(job)\n        except queue.Full:\n            with self.lock:\n                self.pending.pop(req_id, None)\n            raise HTTPException(status_code=503, detail="embedding queue is full")\n\n        self.total_requests += 1\n        self.total_texts += len(texts)\n\n        try:\n            return await asyncio.wait_for(asyncio.wrap_future(fut), timeout=REQUEST_TIMEOUT_S)\n        except asyncio.TimeoutError:\n            with self.lock:\n                self.pending.pop(req_id, None)\n            raise HTTPException(status_code=504, detail="embedding request timed out")\n\n    def stop(self) -> None:\n        for _ in self.workers:\n            try:\n                self.request_q.put_nowait(None)\n            except Exception:\n                pass\n        for p in self.workers:\n            p.join(timeout=5)\n            if p.is_alive():\n                p.terminate()\n\ncoordinator = Coordinator()\n\napp = FastAPI(\n    title="Qwen3-Embedding-4B dual-T4 service",\n    version="1.0.0",\n    default_response_class=ORJSONResponse,\n)\n\ncors = [x.strip() for x in os.getenv("CORS_ORIGINS", "").split(",") if x.strip()]\nif cors:\n    app.add_middleware(\n        CORSMiddleware,\n        allow_origins=cors,\n        allow_credentials=True,\n        allow_methods=["GET", "POST"],\n        allow_headers=["*"],\n    )\n\nasync def require_api_key(\n    authorization: str | None = Header(default=None),\n    x_api_key: str | None = Header(default=None),\n) -> None:\n    if not API_KEY:\n        return\n    supplied = ""\n    if authorization and authorization.lower().startswith("bearer "):\n        supplied = authorization[7:].strip()\n    elif x_api_key:\n        supplied = x_api_key.strip()\n    if supplied != API_KEY:\n        raise HTTPException(status_code=401, detail="invalid API key")\n\n@app.on_event("startup")\nasync def on_startup() -> None:\n    coordinator.start()\n\n@app.on_event("shutdown")\nasync def on_shutdown() -> None:\n    coordinator.stop()\n\n@app.get("/healthz")\nasync def healthz() -> dict[str, Any]:\n    return {\n        "status": "ok",\n        "model": MODEL_ID,\n        "model_source": MODEL_SOURCE,\n        "embedding_dim": EMBEDDING_DIM,\n        "gpu_count": len(GPU_IDS),\n        "ready_gpus": sorted(coordinator.ready_gpus),\n        "failed_workers": coordinator.failed_workers,\n        "max_seq_length": MAX_SEQ_LENGTH,\n        "gpu_batch_texts": GPU_BATCH_TEXTS,\n        "dynamic_batch_wait_ms": DYNAMIC_BATCH_WAIT_MS,\n        "api_key_enabled": bool(API_KEY),\n        "uptime_s": round(time.time() - coordinator.started_at, 2),\n    }\n\n@app.get("/readyz")\nasync def readyz() -> dict[str, Any]:\n    if len(coordinator.ready_gpus) != len(GPU_IDS):\n        raise HTTPException(\n            status_code=503,\n            detail={\n                "ready_gpus": sorted(coordinator.ready_gpus),\n                "expected_gpus": GPU_IDS,\n                "failed": coordinator.failed_workers,\n            },\n        )\n    return {"status": "ready", "gpus": sorted(coordinator.ready_gpus)}\n\n@app.get("/metrics")\nasync def metrics(_: None = Depends(require_api_key)) -> dict[str, Any]:\n    return {\n        "total_requests": coordinator.total_requests,\n        "total_texts": coordinator.total_texts,\n        "pending_requests": len(coordinator.pending),\n        "workers_alive": [p.is_alive() for p in coordinator.workers],\n        "ready_gpus": sorted(coordinator.ready_gpus),\n    }\n\n@app.get("/v1/models")\nasync def list_models(_: None = Depends(require_api_key)) -> dict[str, Any]:\n    return {\n        "object": "list",\n        "data": [{\n            "id": MODEL_ID,\n            "object": "model",\n            "owned_by": "local-kaggle",\n            "embedding_dim": EMBEDDING_DIM,\n            "max_seq_length": MAX_SEQ_LENGTH,\n        }],\n    }\n\nasync def _embed_impl(request: EmbeddingRequest) -> dict[str, Any]:\n    if request.model and request.model != MODEL_ID:\n        raise HTTPException(\n            status_code=400,\n            detail=f"model mismatch: requested={request.model!r}, available={MODEL_ID!r}",\n        )\n\n    texts = _as_texts(request.input)\n    result = await coordinator.submit(texts, request.input_type)\n\n    data = [\n        {"object": "embedding", "index": i, "embedding": vec}\n        for i, vec in enumerate(result["vectors"])\n    ]\n    return {\n        "object": "list",\n        "data": data,\n        "model": MODEL_ID,\n        "usage": {"prompt_tokens": 0, "total_tokens": 0},\n        "meta": {\n            "input_type": request.input_type,\n            "worker": result["worker"],\n            "dynamic_batch_texts": result["batch_texts"],\n            "gpu_ms": round(float(result["gpu_ms"]), 3),\n            "normalized": True,\n            "embedding_dim": EMBEDDING_DIM,\n        },\n    }\n\n@app.post("/v1/embeddings")\nasync def embeddings_v1(\n    request: EmbeddingRequest,\n    _: None = Depends(require_api_key),\n) -> dict[str, Any]:\n    return await _embed_impl(request)\n\n@app.post("/embeddings")\nasync def embeddings_alias(\n    request: EmbeddingRequest,\n    _: None = Depends(require_api_key),\n) -> dict[str, Any]:\n    return await _embed_impl(request)\n\nif __name__ == "__main__":\n    host = os.getenv("EMBED_HOST", "0.0.0.0")\n    port = int(os.getenv("EMBED_PORT", "8001"))\n    uvicorn.run(\n        app,\n        host=host,\n        port=port,\n        workers=1,\n        log_level=os.getenv("LOG_LEVEL", "info"),\n    )\n'
compile(SERVER_CODE, str(SERVER_PATH), "exec")
SERVER_PATH.write_text(SERVER_CODE, encoding="utf-8")
print("Wrote:", SERVER_PATH)
print("Server source lines:", len(SERVER_CODE.splitlines()))


In [ ]:
# 5) Launch server in the background.
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

LOG_PATH = Path("/kaggle/working/qwen3_4b_embedding_server.log")

# Stop an earlier server launched by this notebook, if any.
if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=10)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()

log_handle = LOG_PATH.open("w", encoding="utf-8")
SERVER_PROCESS = subprocess.Popen(
    [sys.executable, str(SERVER_PATH)],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
    start_new_session=True,
)

print("PID:", SERVER_PROCESS.pid)
print("Log:", LOG_PATH)
print("Local base URL:", f"http://127.0.0.1:{PORT}/v1")


In [ ]:
# 6) Wait until both GPU workers are ready.
import json
import time
import httpx

BASE_ROOT = f"http://127.0.0.1:{PORT}"
AUTH_HEADERS = {}
_api_key = os.getenv("EMBED_API_KEY", "").strip()
if _api_key:
    AUTH_HEADERS["Authorization"] = f"Bearer {_api_key}"

deadline = time.time() + 240
last = None
while time.time() < deadline:
    if SERVER_PROCESS.poll() is not None:
        print(LOG_PATH.read_text(encoding="utf-8")[-12000:])
        raise RuntimeError(f"Server exited with code {SERVER_PROCESS.returncode}")
    try:
        r = httpx.get(f"{BASE_ROOT}/readyz", timeout=2)
        last = (r.status_code, r.text)
        if r.status_code == 200:
            print("READY:", r.json())
            break
    except Exception as exc:
        last = repr(exc)
    time.sleep(2)
else:
    print(LOG_PATH.read_text(encoding="utf-8")[-12000:])
    raise TimeoutError(f"Server did not become ready. Last response: {last}")

print("HEALTH:")
print(json.dumps(httpx.get(f"{BASE_ROOT}/healthz", timeout=5).json(), indent=2, ensure_ascii=False))


In [ ]:
# 7) API contract + vector validation.
import math
import httpx

def embed(texts, input_type="query"):
    payload = {
        "model": MODEL_ID,
        "input": texts,
        "input_type": input_type,
    }
    r = httpx.post(
        f"{BASE_ROOT}/v1/embeddings",
        headers=AUTH_HEADERS,
        json=payload,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()

models = httpx.get(f"{BASE_ROOT}/v1/models", headers=AUTH_HEADERS, timeout=10)
models.raise_for_status()
print("MODELS:", models.json())

query_body = embed("người đàn ông nói về trí tuệ nhân tạo", input_type="query")
doc_body = embed([
    "Một người đàn ông đang thuyết trình về trí tuệ nhân tạo.",
    "Khán giả vỗ tay sau bài phát biểu."
], input_type="document")

qvec = query_body["data"][0]["embedding"]
assert len(qvec) == EXPECTED_DIM
qnorm = math.sqrt(sum(float(x) * float(x) for x in qvec))
assert abs(qnorm - 1.0) <= 1e-2, qnorm

assert all(len(item["embedding"]) == EXPECTED_DIM for item in doc_body["data"])

print("query dim:", len(qvec), "norm:", round(qnorm, 6))
print("query meta:", query_body["meta"])
print("document meta:", doc_body["meta"])
print("API contract OK")


## Query vs document mode

This distinction matters for Qwen3 embeddings:

- **ASR segments inserted into Milvus** → `input_type="document"`
- **User search query from the web** → `input_type="query"` (default)

The query path uses Qwen's retrieval query prompt. The document path does not.

Therefore your offline ASR vectors and online query vectors remain asymmetric in the intended retrieval way.


In [ ]:
# 8) Local concurrency benchmark.
# This measures your actual Kaggle session instead of assuming an RPS number.
import asyncio
import json
import statistics
import time
import httpx

N_REQUESTS = 64
CONCURRENCY = 16
QUERIES = [
    "người phụ nữ đang nói trên sân khấu",
    "ai đang nhắc đến một con số",
    "đoạn hội thoại nói về trường học",
    "người dẫn chương trình giới thiệu khách mời",
    "bài phát biểu về công nghệ",
    "người đàn ông nói về một sự kiện",
    "khán giả cười trong lúc trò chuyện",
    "ai đó đọc một tiêu đề hoặc tên riêng",
]

sem = asyncio.Semaphore(CONCURRENCY)

async def one(client, idx):
    payload = {"model": MODEL_ID, "input": QUERIES[idx % len(QUERIES)], "input_type": "query"}
    async with sem:
        t0 = time.perf_counter()
        r = await client.post(f"{BASE_ROOT}/v1/embeddings", headers=AUTH_HEADERS, json=payload)
        r.raise_for_status()
        body = r.json()
        return (time.perf_counter() - t0) * 1000, body["meta"]

async def run_bench():
    timeout = httpx.Timeout(120)
    limits = httpx.Limits(max_connections=CONCURRENCY, max_keepalive_connections=CONCURRENCY)
    async with httpx.AsyncClient(timeout=timeout, limits=limits) as client:
        t0 = time.perf_counter()
        rows = await asyncio.gather(*(one(client, i) for i in range(N_REQUESTS)))
        wall = time.perf_counter() - t0
    lat = [x[0] for x in rows]
    workers = {}
    dynamic_batches = []
    for _, meta in rows:
        workers[meta["worker"]] = workers.get(meta["worker"], 0) + 1
        dynamic_batches.append(meta["dynamic_batch_texts"])
    return {
        "requests": N_REQUESTS,
        "concurrency": CONCURRENCY,
        "wall_seconds": round(wall, 3),
        "requests_per_second": round(N_REQUESTS / wall, 3),
        "latency_ms_p50": round(statistics.median(lat), 3),
        "latency_ms_p95": round(sorted(lat)[max(0, math.ceil(0.95 * len(lat)) - 1)], 3),
        "latency_ms_p99": round(sorted(lat)[max(0, math.ceil(0.99 * len(lat)) - 1)], 3),
        "worker_distribution": workers,
        "mean_dynamic_batch_texts": round(statistics.mean(dynamic_batches), 3),
    }

BENCHMARK = await run_bench()
print(json.dumps(BENCHMARK, indent=2, ensure_ascii=False))


## Tuning rules

Start with the defaults, then use the benchmark cell to tune:

- `QWEN_GPU_BATCH_TEXTS=8`: conservative starting point for T4.
- `QWEN_BATCH_WAIT_MS=8`: lets concurrent short search requests coalesce without adding too much latency.
- `QWEN_EMBED_MAX_SEQ_LEN=2048`: matches the existing ASR embedding notebook and avoids paying the 32K-context cost for short transcripts.
- one process per GPU: two model replicas for two T4s.
- do **not** increase Uvicorn workers; increase HTTP concurrency at the client/coordinator level instead.
- if you see CUDA OOM, lower `QWEN_GPU_BATCH_TEXTS` to 4. The worker also includes an automatic split fallback.
- if GPU utilization is low while request concurrency is high, try batch size 12 or 16 and benchmark again.

There is no trustworthy fixed RPS value without measuring your actual average query length, package versions,
Kaggle host CPU, and current GPU session. Use the benchmark output as the source of truth.


In [ ]:
# 9) Optional secure ngrok tunnel for your external web/backend.
# Required Kaggle Secrets:
#   NGROK_AUTHTOKEN
#   EMBED_API_KEY
#
# Do not expose the endpoint publicly without an API key.
import os

if not os.getenv("EMBED_API_KEY", "").strip():
    raise RuntimeError(
        "EMBED_API_KEY is empty. Add a Kaggle Secret named EMBED_API_KEY before opening a public tunnel."
    )

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "").strip()
    if not NGROK_AUTHTOKEN:
        NGROK_AUTHTOKEN = _secrets.get_secret("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN", "").strip()

if not NGROK_AUTHTOKEN:
    raise RuntimeError("Add Kaggle Secret NGROK_AUTHTOKEN.")

from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTHTOKEN

# Close older tunnels opened from this runtime.
try:
    ngrok.kill()
except Exception:
    pass

TUNNEL = ngrok.connect(addr=PORT, proto="http")
PUBLIC_ROOT = TUNNEL.public_url.rstrip("/")
PUBLIC_V1_BASE_URL = PUBLIC_ROOT + "/v1"

print("Public root:", PUBLIC_ROOT)
print("Use in backend as EMBEDDING_BASE_URL:", PUBLIC_V1_BASE_URL)
print("Model:", MODEL_ID)
print("Embedding dim:", EXPECTED_DIM)


In [ ]:
# 10) Verify the public tunnel using the same API contract.
if "PUBLIC_ROOT" not in globals():
    raise RuntimeError("Run the ngrok cell first.")

public_models = httpx.get(
    f"{PUBLIC_ROOT}/v1/models",
    headers=AUTH_HEADERS,
    timeout=30,
)
public_models.raise_for_status()

public_embed = httpx.post(
    f"{PUBLIC_ROOT}/v1/embeddings",
    headers=AUTH_HEADERS,
    json={
        "model": MODEL_ID,
        "input": "đoạn video có người đang nói về công nghệ",
        "input_type": "query",
    },
    timeout=60,
)
public_embed.raise_for_status()
vec = public_embed.json()["data"][0]["embedding"]

print("public models:", public_models.json())
print("public vector dim:", len(vec))
assert len(vec) == EXPECTED_DIM


## Web/backend integration

The repository's existing embedding clients use:

```python
response = client.post(
    f"{base_url.rstrip('/')}/embeddings",
    json={"model": model_name, "input": query},
)
```

Therefore set:

```env
EMBEDDING_BASE_URL=<PUBLIC_ROOT>/v1
EMBEDDING_MODEL=Qwen/Qwen3-Embedding-4B
EMBEDDING_DIM=2560
```

If `EMBED_API_KEY` is enabled, add this header from the backend:

```http
Authorization: Bearer <EMBED_API_KEY>
```

For this server, omitting `input_type` defaults to `query`, which keeps the current search-client payload compatible.
When you use the endpoint for ASR ingestion/document embedding, explicitly send `"input_type": "document"`.


## Milvus compatibility — important

The repository's `MilvusTextEmbeddingSink` creates a collection using the vector dimension observed on the first insert.

`Qwen3-Embedding-4B` returns **2560-dimensional** vectors. If your existing
`text_embeddings_vietnamese` collection was created with another embedding model/dimension, **do not insert Qwen
vectors into that existing collection**.

Recommended new collection:

```text
text_embeddings_qwen3_4b_v1
```

Store the model version as:

```text
Qwen/Qwen3-Embedding-4B
```

Both offline ASR document vectors and online query vectors must come from the same Qwen embedding model and
normalization setup.


In [ ]:
# 11) Optional Milvus/Zilliz semantic-search smoke test.
# Expected Kaggle Secrets or environment variables:
#   MILVUS_URI
#   MILVUS_TOKEN
#
# Run only after you have uploaded 2560-d Qwen ASR embeddings.
import os

MILVUS_COLLECTION = os.getenv("ASR_QWEN_MILVUS_COLLECTION", "text_embeddings_qwen3_4b_v1")
MILVUS_URI = os.getenv("MILVUS_URI", "").strip()
MILVUS_TOKEN = os.getenv("MILVUS_TOKEN", "").strip()

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    if not MILVUS_URI:
        try: MILVUS_URI = _s.get_secret("MILVUS_URI")
        except Exception: pass
    if not MILVUS_TOKEN:
        try: MILVUS_TOKEN = _s.get_secret("MILVUS_TOKEN")
        except Exception: pass
except Exception:
    pass

if not MILVUS_URI:
    print("MILVUS_URI not configured; skipping Milvus smoke test.")
else:
    from pymilvus import MilvusClient

    q = embed("người đang nói về trí tuệ nhân tạo", input_type="query")
    qvec = q["data"][0]["embedding"]
    assert len(qvec) == EXPECTED_DIM

    client = MilvusClient(uri=MILVUS_URI, token=MILVUS_TOKEN or None)
    results = client.search(
        collection_name=MILVUS_COLLECTION,
        data=[qvec],
        limit=10,
        output_fields=["video_id", "text", "doc_type", "model_version"],
    )
    print("Collection:", MILVUS_COLLECTION)
    print(json.dumps(results[0][:5] if results else [], ensure_ascii=False, indent=2, default=str))


In [ ]:
# 12) Operational inspection.
print("Server metrics:")
print(json.dumps(
    httpx.get(f"{BASE_ROOT}/metrics", headers=AUTH_HEADERS, timeout=10).json(),
    indent=2,
    ensure_ascii=False
))
print("\nLast server log lines:")
print(LOG_PATH.read_text(encoding="utf-8")[-8000:])


In [ ]:
# 13) Cleanup when you are done.
# Do not run this cell while your web is still using the server.
try:
    if "TUNNEL" in globals():
        from pyngrok import ngrok
        ngrok.disconnect(TUNNEL.public_url)
except Exception as exc:
    print("Tunnel cleanup:", exc)

if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=15)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()

print("Stopped.")
